# B: sample shuffled trajectories (shuffled laminar and shuffled still air)

Fills layer `trajectories` of the figure SVG: 18 flash-trial trajectories per shuffled dataset, 3 columns x 6 rows.
Pre-flash in gray, during the flash in red, after the flash in black (arrowhead at the end), up to 4 s after the flash.

Plotting functions and trajectory selection come from `time_shift_flies/plot_shuffled_trajectories.py`. With the same seed,
these are the same trajectories as the SHUFFLED panels of `time_shift_flies/shuffled_trajectories.svg`. Shuffling keeps
each trajectory's ground speed, timing and NaN pattern, so the selection on the shuffled data matches the selection on the
original data made by that script.

Each trajectory is centered on its own bounding box. Each panel has its own scale (scale bar under the bottom-right
trajectory), sized so every trajectory in the panel fits.

Run A → B → C → D (any order; each fills its own layer).

In [ ]:
import matplotlib
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.weight'] = 'normal'
from splitflow.figure_style import apply_house_style  # Helvetica 6 pt, Helvetica mathtext

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt

import figurefirst as fifi

from scenarios import FIGURE_NAME, get_trajec_filename, save_layer

sys.path.insert(0, 'time_shift_flies')
import plot_shuffled_trajectories as pst  # sets its own rcParams on import, so apply the house style after

apply_house_style(size=matplotlib.rcParams["font.size"])

In [ ]:
fifi_figure_label = 'trajectories'
PANELS = ['shuffled_laminar', 'shuffled_stillair']  # order matters for the random draw (same as plot_shuffled_trajectories.py)
SEED = 42  # plot_shuffled_trajectories.py default

GAP_X_MM = 1.0  # gap between trajectory columns
GAP_Y_MM = 0.6  # gap between trajectory rows
SCALE_BAR_FONTSIZE = 6

In [ ]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [ ]:
rng = np.random.default_rng(SEED)
fig = layout.figures[fifi_figure_label]
fig_w_mm, fig_h_mm = np.array(fig.get_size_inches()) * 25.4

for key in PANELS:
    df = pst.load(get_trajec_filename(key))  # flash trials only
    ids = pst.qualified_ids(df)
    n = min(pst.N_TRAJS, len(ids))
    chosen = rng.choice(ids, size=n, replace=False)
    trajs = [pst.get_trajectory(df, tid) for tid in chosen]
    hw = pst.fit_half_width(trajs)
    print('%-18s qualified = %4d  plotted = %d  window = +/-%.2f x +/-%.2f m' % (key, len(ids), n, hw, hw / 2))

    # split the template axis into a grid of trajectory axes
    parent = layout.axes[(fifi_figure_label, key)]
    parent.axis('off')
    pos = parent.get_position()
    gap_x, gap_y = GAP_X_MM / fig_w_mm, GAP_Y_MM / fig_h_mm
    w = (pos.width - (pst.N_TRAJ_COLS - 1) * gap_x) / pst.N_TRAJ_COLS
    h = (pos.height - (pst.N_TRAJ_ROWS - 1) * gap_y) / pst.N_TRAJ_ROWS
    for j, t in enumerate(trajs):
        r, c = divmod(j, pst.N_TRAJ_COLS)
        ax = fig.add_axes([pos.x0 + c * (w + gap_x), pos.y1 - (r + 1) * h - r * gap_y, w, h])
        pst.plot_trajectory(t, ax, hw)
    pst.add_scale_bar(ax, hw, fontsize=SCALE_BAR_FONTSIZE)  # under the bottom-right trajectory

In [ ]:
layout.apply_svg_attrs()
save_layer(layout, fifi_figure_label)